# ML Pipelines, Reproducibility & Versioning

Understanding ML pipelines from first principles — not just Airflow syntax — is what separates MLE candidates from data scientists. This note builds a mini pipeline framework from scratch to demystify DAG-based orchestration, then covers versioning and experiment tracking.

## What Interviewers Test
- DAG thinking: tasks, dependencies, topological execution
- Why "it works in the notebook" fails in production
- Model and data versioning: what to version and why
- Experiment tracking: what to log and how
- Config management and reproducibility

In [ ]:
import hashlib, json, time, os
from collections import defaultdict
import numpy as np

# --- Mini pipeline framework (~50 lines) ---
class Task:
    def __init__(self, name, fn, inputs=None):
        self.name   = name
        self.fn     = fn
        self.inputs = inputs or []   # names of upstream tasks

class Pipeline:
    def __init__(self):
        self.tasks  = {}   # name → Task
        self._cache = {}   # name → (input_hash, output)

    def add(self, task):
        self.tasks[task.name] = task
        return self

    def _topo_sort(self):
        """Kahn's algorithm for topological sort."""
        in_degree = {n: 0 for n in self.tasks}
        for task in self.tasks.values():
            for inp in task.inputs:
                in_degree[task.name] += 1
                # Note: in_degree tracks how many upstream tasks each task has
        # Rebuild: count predecessors per task
        in_degree = defaultdict(int)
        for task in self.tasks.values():
            in_degree[task.name]  # ensure exists
            for inp in task.inputs:
                in_degree[task.name] += 1

        queue = [n for n, d in in_degree.items() if d == 0]
        order = []
        while queue:
            n = queue.pop(0); order.append(n)
            for task in self.tasks.values():
                if n in task.inputs:
                    in_degree[task.name] -= 1
                    if in_degree[task.name] == 0:
                        queue.append(task.name)
        if len(order) != len(self.tasks):
            raise ValueError("Cycle detected in pipeline!")
        return order

    def _hash_inputs(self, task_name, outputs):
        """Hash the inputs of a task for cache key."""
        input_data = {n: str(outputs.get(n, '')) for n in self.tasks[task_name].inputs}
        return hashlib.md5(json.dumps(input_data, sort_keys=True).encode()).hexdigest()

    def run(self, use_cache=True):
        outputs = {}
        for name in self._topo_sort():
            task = self.tasks[name]
            kwargs = {inp: outputs[inp] for inp in task.inputs if inp in outputs}
            input_hash = self._hash_inputs(name, outputs)
            if use_cache and name in self._cache and self._cache[name][0] == input_hash:
                print(f"  CACHE HIT: {name}")
                outputs[name] = self._cache[name][1]
            else:
                print(f"  RUNNING:   {name}")
                t0 = time.time()
                result = task.fn(**kwargs)
                print(f"             took {(time.time()-t0)*1000:.1f}ms")
                self._cache[name] = (input_hash, result)
                outputs[name] = result
        return outputs

# --- Define a simple ML pipeline ---
np.random.seed(42)

def load_data():
    X = np.random.randn(500, 5)
    y = (X[:,0] + X[:,1] > 0).astype(int)
    return X, y

def preprocess(raw_data):
    X, y = raw_data
    mu, std = X.mean(axis=0), X.std(axis=0) + 1e-8
    return (X - mu) / std, y, mu, std

def train_model(preprocessed):
    from sklearn.linear_model import LogisticRegression
    X, y, mu, std = preprocessed
    model = LogisticRegression(max_iter=200).fit(X, y)
    return model, mu, std

def evaluate(model_artifacts, raw_data):
    from sklearn.metrics import roc_auc_score
    model, mu, std = model_artifacts
    X, y = raw_data
    X_norm = (X - mu) / std
    return roc_auc_score(y, model.predict_proba(X_norm)[:,1])

p = Pipeline()
p.add(Task('load',       load_data))
p.add(Task('preprocess', preprocess, inputs=['load']))
p.add(Task('train',      train_model, inputs=['preprocess']))
p.add(Task('evaluate',   evaluate, inputs=['train', 'load']))

print("=== First run ===")
out = p.run()
print(f"\nAUC: {out['evaluate']:.4f}")
print("\n=== Second run (cache hits) ===")
out2 = p.run()


## Model Registry Pattern


In [ ]:
import json, os, hashlib
from datetime import datetime

class ModelRegistry:
    """Toy model registry — tracks versions, stages, and metadata."""
    def __init__(self, registry_dir='/tmp/model_registry'):
        self.dir = registry_dir
        os.makedirs(registry_dir, exist_ok=True)
        self.index_path = os.path.join(registry_dir, 'index.json')
        self.index = json.load(open(self.index_path)) if os.path.exists(self.index_path) else {}

    def _save_index(self):
        with open(self.index_path, 'w') as f:
            json.dump(self.index, f, indent=2)

    def register(self, model_name, run_id, metrics, params, stage='development'):
        version = len(self.index.get(model_name, [])) + 1
        entry = {
            'version': version,
            'run_id': run_id,
            'metrics': metrics,
            'params': params,
            'stage': stage,
            'registered_at': datetime.utcnow().isoformat(),
        }
        self.index.setdefault(model_name, []).append(entry)
        self._save_index()
        print(f"Registered {model_name} v{version} (stage={stage})")
        return version

    def promote(self, model_name, version, new_stage):
        for entry in self.index.get(model_name, []):
            if entry['version'] == version:
                entry['stage'] = new_stage
                self._save_index()
                print(f"Promoted {model_name} v{version} → {new_stage}")
                return
        raise ValueError(f"Version {version} not found")

    def get_latest(self, model_name, stage='production'):
        candidates = [e for e in self.index.get(model_name, []) if e['stage'] == stage]
        return max(candidates, key=lambda e: e['version']) if candidates else None

# Usage
reg = ModelRegistry()
v1 = reg.register('ctr_model', run_id='run_001',
    metrics={'auc': 0.82, 'log_loss': 0.31},
    params={'lr': 0.1, 'n_estimators': 100})
v2 = reg.register('ctr_model', run_id='run_002',
    metrics={'auc': 0.85, 'log_loss': 0.28},
    params={'lr': 0.05, 'n_estimators': 200})

reg.promote('ctr_model', 2, 'production')
latest = reg.get_latest('ctr_model', stage='production')
print(f"\nLatest production model: v{latest['version']}, AUC={latest['metrics']['auc']}")


## Experiment Tracker


In [ ]:
class ExperimentTracker:
    """Minimal tracker — logs JSON runs."""
    def __init__(self, log_dir='/tmp/experiments'):
        os.makedirs(log_dir, exist_ok=True)
        self.log_dir = log_dir
        self.current_run = None

    def start_run(self, run_name=None):
        self.current_run = {
            'run_id': hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
            'run_name': run_name or 'run',
            'started_at': datetime.utcnow().isoformat(),
            'params': {}, 'metrics': {}, 'artifacts': []
        }
        return self.current_run['run_id']

    def log_param(self, key, value):
        self.current_run['params'][key] = value

    def log_metric(self, key, value, step=None):
        entry = {'value': value}
        if step is not None: entry['step'] = step
        self.current_run['metrics'].setdefault(key, []).append(entry)

    def end_run(self):
        path = os.path.join(self.log_dir, f"{self.current_run['run_id']}.json")
        self.current_run['ended_at'] = datetime.utcnow().isoformat()
        with open(path, 'w') as f:
            json.dump(self.current_run, f, indent=2)
        print(f"Run saved: {path}")
        return self.current_run

tracker = ExperimentTracker()
rid = tracker.start_run('lr_baseline')
tracker.log_param('learning_rate', 0.01)
tracker.log_param('n_epochs', 100)
for step in range(5):
    tracker.log_metric('train_loss', 1.0 - step * 0.15, step=step)
    tracker.log_metric('val_loss',   1.1 - step * 0.12, step=step)
run = tracker.end_run()
print(f"Logged {len(run['metrics'])} metric series, {len(run['params'])} params")


## Common Interview Questions

**Q: Why does "it works in the notebook" fail in production?**
Notebooks have hidden state (out-of-order cell execution), implicit dependencies (global variables), no version control integration, and no logging or retry logic. Production pipelines need deterministic execution order, explicit dependency declarations, failure handling, and audit logs.

**Q: What is a DAG in the context of ML pipelines?**
A Directed Acyclic Graph where nodes are tasks and edges are dependencies. Topological sort gives a valid execution order. Tools like Airflow, Prefect, and Dagster manage DAG scheduling, retries, backfilling, and dependency resolution. The acyclic constraint ensures no circular dependencies.

**Q: What should a model registry track?**
Model version, training run ID (linking to experiment tracker), metrics on held-out eval set, hyperparameters and training config, training data version/hash, deployment stage (dev/staging/production), and promotion history. This enables rollback, audit, and comparison across versions.

**Q: What is the purpose of input hashing in a pipeline cache?**
Cache hits are only valid if the inputs haven't changed. By hashing the outputs of upstream tasks, we can detect when any input to a task has changed — then invalidate its cache and rerun it, even if the task's code hasn't changed. This is how Makefile-style incremental builds work.

## Key Takeaways
- DAG pipelines: tasks declare dependencies; topological sort gives execution order; cache by input hash
- Model registry: version models with metrics, params, stage, training run link — enables rollback
- Experiment tracker: log all hyperparams and metrics per run — prevents "which config got that AUC?" confusion
- Config management: all hyperparams in config files, never hardcoded — enables reproducibility
- "Works in notebook" → fails in production: hidden state, no versioning, no retry, no logging
- Data versioning: hash the input data, log the hash with each experiment — enables exact reproduction